# 3D Gaussian Splatting for sparse-view CT

## 0. Setup

In [1]:
# Mount Google Drive (data is NOT in git – too large)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists("/content/MGST_5_IPPR_CT_Reconstruction"):
        !git clone -q -b baseline-sophia https://github.com/gs0232/IPPR_CT_Reconstruction.git /content/MGST_5_IPPR_CT_Reconstruction
    sys.path.insert(0, "/content/MGST_5_IPPR_CT_Reconstruction/Baselines")
else:
    sys.path.insert(0, ".")
    sys.path.insert(0, "../Baselines")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Runtime -> Change runtime type -> GPU
import math, re, time
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from PIL import Image
from skimage.transform import iradon, radon
from IPython.display import display

import metrics   # our shared metric functions (PSNR, SSIM, RMSE, head mask)
metrics._self_test()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "NO GPU!")

metrics.py self-test passed: {'psnr': 26.054, 'ssim': 0.301, 'rmse': 0.05, 'psnr_head': 26.124, 'ssim_head': 0.403, 'rmse_head': 0.049}
device: cuda | Tesla T4


In [ ]:
# protocol constants: copied from 02_fbp_baseline.ipynb
N_ANGLES    = 984          # projection angles of a full scan
N_DET       = 888          # detector pixels
ANGLE_RANGE = 360          # degrees covered by the 984 angles
OUTPUT_SIZE = 888          # reconstruction size
FILTER      = "ramp"       # FBP filter
SLICE_STEP  = 4            # slices 00, 04, 08, ... have a ground truth

THETA_FULL_DEG = np.linspace(0, ANGLE_RANGE, N_ANGLES, endpoint=False)


def view_indices(n_views):
    """Same angle selection as the baseline (spread over the first 180 deg only,
    because theta and theta+180 carry the same information)."""
    n_half = N_ANGLES if ANGLE_RANGE == 180 else N_ANGLES // 2
    return np.round(np.linspace(0, n_half, n_views, endpoint=False)).astype(int)


In [ ]:
@dataclass
class Cfg:
    # paths
    data_root: str = "/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction"
    sparse_png_dir: str = "content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/sparse_output"  
    gt_png_dir: str = "content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/full_output"      
    out_dir: str = "content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/outputs/3dgs"        
    # what to run 
    series: list = field(default_factory=list)   # series to reconstruct, e.g. ["series_1040", ...]
    slice_step: int = SLICE_STEP                 # only slices 00, 04, 08, ... (those with a ground truth)
    max_slices: int = 0                          # 0 = all of them
    seed: int = 42
    n_views_input: int = 98                      # views behind the sparse PNGs -> 'v098' in the file names

    # model / init 
    init: str = "image"      # "image" (Gaussians sampled from the sparse PNG) | "random"  
    n_init: int = 15000      
    max_gaussians: int = 10000  
    init_gamma: float = 0.5    
    init_uniform_frac: float = 0.1
    s_min: float = 0.5       # px, clamp on Gaussian scale
    s_max: float = 60.0
    angle_sign: float = 1.0  # -1 only if the geometry check says so

    # optimisation 
    iters: int = 500       
    views_per_step: int = 8    
    gauss_chunk: int = 16384    
    use_checkpoint: bool = True
    lr_xy: float = 0.1       # pixels per step (Adam)
    lr_scale: float = 0.01
    lr_rot: float = 0.01
    lr_rho: float = 0.02
    lr_final_ratio: float = 0.05   # lr decays exponentially to this fraction
    tv_weight: float = 0.01  # 0 disables TV
    tv_patch: int = 16    
    tv_chunk: int = 4096
    l2_weight: float = 0.0   

    # ---- adaptive density control -----------------------------------------------------
    densify_from: float = 0.10    # fractions of `iters`
    densify_until: float = 0.60
    densify_every: int = 100
    densify_top_frac: float = 0.05
    split_px: float = 4.0         
    prune_rho_rel: float = 0.003  # prune if density < this * tissue-density reference

    # ---- evaluation / io ------------------------------------------------------------------
    view_synthesis: bool = True   # also score projections at held-out angles (vs. projections of the GT image)
    n_heldout: int = 64
    save_preview: bool = True
    log_every: int = 250
    smoke: bool = False
    tag: str = ""            # non-empty -> outputs/CSV go to <out_dir>/_<tag>/ (tuning runs never touch the official outputs)

    def finalize(self):
        root = Path(self.data_root)
        self.sparse_png_dir = self.sparse_png_dir or str(root / "sparse_output")
        self.gt_png_dir = self.gt_png_dir or str(root / "full_output")
        self.out_dir = self.out_dir or str(root / "outputs" / "3dgs")
        return self

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction"   

cfg = Cfg(
    data_root      = DATA_ROOT,
    sparse_png_dir = f"{DATA_ROOT}/sparse_output",      # INPUT: sparse-view FBP PNGs
    gt_png_dir     = f"{DATA_ROOT}/full_output",        # ground truth PNGs (evaluation only)
    out_dir        = f"{DATA_ROOT}/outputs/3dgs",       # OUTPUT: series_XXXX_sliceYY_v098.png
)

VAL_SERIES  = [f"series_{i}" for i in range(1050, 1069)]   # validation patients (group decision) -> tuning ONLY
TEST_SERIES = [f"series_{i}" for i in range(1030, 1039)]     # the final test series 

cfg.series = TEST_SERIES
cfg.angle_sign = 1.0       # based on the geometry check 
cfg.finalize()
print("input  <-", cfg.sparse_png_dir)
print("output ->", cfg.out_dir)

input  <- /content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/sparse_output
output -> /content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/outputs/3dgs


## 2. Data helpers: PNG in, PNG out

In [ ]:
def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def circle_mask(size=OUTPUT_SIZE):
    """Reconstruction circle of iradon(circle=True): x^2 + y^2 <= 444^2."""
    c = size // 2
    yy, xx = np.mgrid[:size, :size] - c
    return (xx ** 2 + yy ** 2) <= c ** 2


def slice_index(path):
    return int(re.search(r"(\d+)", Path(path).stem).group(1))      # 'slice_08.png' -> 8


def load_png(path):
    """8-bit grey (RGBA / RGB / L) PNG -> float32 in [0, 1] (same as the team's to_gray)."""
    a = np.array(Image.open(path))
    if a.ndim == 3:
        a = a[..., 0]
    return a.astype(np.float32) / 255.0


def save_png(path, img01):
    """float [0,1] -> 8-bit greyscale PNG (the group's loader accepts L as well as RGBA)."""
    arr = np.clip(np.rint(np.asarray(img01) * 255.0), 0, 255).astype(np.uint8)
    Image.fromarray(arr).save(path)


def list_slices_png(cfg, sid):
    """[(slice_no, sparse_png, gt_png)] of a series: slices on the evaluation grid (00, 04, 08, ...)
    that exist in BOTH sparse_output and full_output."""
    sp, gp = Path(cfg.sparse_png_dir) / sid, Path(cfg.gt_png_dir) / sid
    out = []
    for f in sorted(sp.glob("*.png"), key=slice_index):
        s = slice_index(f)
        if s % cfg.slice_step == 0 and (gp / f.name).exists():
            out.append((s, f, gp / f.name))
    return out


def background_level(img01):
    """The PNGs are min-max normalised FBP images, so 'zero attenuation' (air) is NOT 0 but a
    constant grey (16/255 = 0.063 in the example slice). Median inside the circle = air."""
    return float(np.median(img01[circle_mask(img01.shape[0])]))


def png_to_sinogram(img01, theta_deg):
    """sparse PNG -> (b, img, sino)
    b    : background level of the PNG
    img  : background-removed, non-negative image inside the circle (what the Gaussians must explain)
    sino : its parallel-beam sinogram at the given angles, shape (N_DET, n_angles), in PNG units.
    The PNG is itself an FBP of the measured sinogram, so radon(PNG) ~ the measured sparse projections."""
    assert img01.shape == (OUTPUT_SIZE, OUTPUT_SIZE), f"unexpected PNG size {img01.shape}"
    b = background_level(img01)
    img = (np.clip(img01 - b, 0.0, None) * circle_mask(img01.shape[0])).astype(np.float32)
    sino = radon(img, theta=theta_deg, circle=True).astype(np.float32)
    return b, img, sino

In [ ]:
for p in [cfg.sparse_png_dir, cfg.gt_png_dir, Path(cfg.out_dir).parent]:
    print(os.path.exists(p), p)
print("methods already in outputs/:", os.listdir(Path(cfg.out_dir).parent))
missing = [s for s in VAL_SERIES if not (Path(cfg.sparse_png_dir) / s).exists()]
print("validation series missing in sparse_output:", missing or "none")
if not missing:
    print("slices with input AND ground truth (every 4th):", {s: len(list_slices_png(cfg, s)) for s in VAL_SERIES[:3]})

True /content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/sparse_output
True /content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/full_output
True /content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction/outputs
methods already in outputs/: ['nerf', 'unet', '3dgs']
validation series missing in sparse_output: none
slices with input AND ground truth (every 4th): {'series_1050': 9, 'series_1051': 11, 'series_1052': 10}


## 3. The Gaussian model

In [8]:
TWO_PI = 2.0 * math.pi
PIX_VAR = 1.0 / 12.0   # variance of a 1-pixel-wide box -> detector bin integration


def _project_chunk(xy, log_scale, rot, log_rho, theta, det, s_min, s_max):
    """Project a set of Gaussians for several views.

    xy (n,2), log_scale (n,2), rot (n,), log_rho (n,)
    theta (V,) radians, det (D,) detector coordinates in pixels
    returns (V, D) projections.
    """
    s = torch.exp(log_scale).clamp(s_min, s_max)           # (n,2)
    rho = torch.exp(log_rho)                                # (n,)
    s1, s2 = s[:, 0], s[:, 1]

    cos_t = torch.cos(theta)[:, None]                       # (V,1)
    sin_t = torch.sin(theta)[:, None]
    m = xy[:, 0][None, :] * cos_t - xy[:, 1][None, :] * sin_t   # (V,n) centre on detector

    phi = theta[:, None] + rot[None, :]                     # (V,n)
    var = (s1 ** 2)[None, :] * torch.cos(phi) ** 2 \
        + (s2 ** 2)[None, :] * torch.sin(phi) ** 2 + PIX_VAR    # (V,n)
    amp = (rho * TWO_PI * s1 * s2)[None, :] / torch.sqrt(TWO_PI * var)  # (V,n)

    diff = det[None, None, :] - m[:, :, None]               # (V,n,D)
    prof = torch.exp(-0.5 * diff * diff / var[:, :, None])  # (V,n,D)
    return torch.bmm(amp.unsqueeze(1), prof).squeeze(1)     # (V,D) = sum_n amp * prof

In [ ]:
class GaussianCT(nn.Module):
    def __init__(self, xy, log_scale, rot, log_rho, size=888, s_min=0.5, s_max=60.0):
        super().__init__()
        self.xy = nn.Parameter(xy.contiguous().float())               # (N,2)
        self.log_scale = nn.Parameter(log_scale.contiguous().float()) # (N,2)
        self.rot = nn.Parameter(rot.contiguous().float())             # (N,)
        self.log_rho = nn.Parameter(log_rho.contiguous().float())     # (N,)
        self.size = size
        self.c = size // 2
        self.s_min = float(s_min)
        self.s_max = float(s_max)

    # convenience 
    @property
    def n(self):
        return self.xy.shape[0]

    def scales(self):
        return torch.exp(self.log_scale).clamp(self.s_min, self.s_max)

    def rho(self):
        return torch.exp(self.log_rho)

    # forward model: Gaussians -> sinogram columns 
    def project(self, theta, det, chunk=16384, use_checkpoint=True):
        """theta (V,) radians (angle_sign already applied), det (D,) -> (V,D).

        chunk: Gaussians per chunk. With use_checkpoint=True every chunk is
        recomputed in the backward pass, so peak memory is bounded by
        V * chunk * D floats instead of V * N * D.
        """
        args = (self.xy, self.log_scale, self.rot, self.log_rho)
        if chunk is None or chunk >= self.n:
            return _project_chunk(*args, theta, det, self.s_min, self.s_max)
        out = 0.0
        for i in range(0, self.n, chunk):
            sl = slice(i, i + chunk)
            a = tuple(p[sl] for p in args)
            if use_checkpoint and torch.is_grad_enabled():
                out = out + checkpoint(_project_chunk, *a, theta, det,
                                       self.s_min, self.s_max, use_reentrant=False)
            else:
                out = out + _project_chunk(*a, theta, det, self.s_min, self.s_max)
        return out

    # image-domain rendering (evaluation + TV regulariser) 
    def render_region(self, ys, xs, chunk=64):
        """Evaluate the density on the grid ys (H,) x xs (W,) (pixel offsets from
        the centre, float tensors). Differentiable. Returns (H, W)."""
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        gx, gy = gx.reshape(-1), gy.reshape(-1)
        s, rho = self.scales(), self.rho()
        cp, sp = torch.cos(self.rot), torch.sin(self.rot)
        out = torch.zeros_like(gx)
        for i in range(0, self.n, chunk):
            sl = slice(i, i + chunk)
            dx = gx[None, :] - self.xy[sl, 0:1]                      # (c,P)
            dy = gy[None, :] - self.xy[sl, 1:2]
            a = (dx * cp[sl, None] + dy * sp[sl, None]) / s[sl, 0:1]   # along e1
            b = (-dx * sp[sl, None] + dy * cp[sl, None]) / s[sl, 1:2]  # along e2
            out = out + (rho[sl, None] * torch.exp(-0.5 * (a * a + b * b))).sum(0)
        inside = (gx ** 2 + gy ** 2) <= self.c ** 2   # same circle as iradon(circle=True)
        return (out * inside).view(len(ys), len(xs))

    @torch.no_grad()
    def render_image(self, chunk=64):
        """Full size x size reconstruction in the model's (normalised) units."""
        dev = self.xy.device
        co = torch.arange(self.size, device=dev, dtype=torch.float32) - self.c
        return self.render_region(co, co, chunk=chunk)

    # adaptive density control (clone / split / prune)
    def _set(self, xy, log_scale, rot, log_rho):
        self.xy = nn.Parameter(xy.contiguous())
        self.log_scale = nn.Parameter(log_scale.contiguous())
        self.rot = nn.Parameter(rot.contiguous())
        self.log_rho = nn.Parameter(log_rho.contiguous())

    @torch.no_grad()
    def densify_and_prune(self, grad_accum, denom, top_frac, split_px, rho_prune, max_n):
        """Standard 3DGS-style control, simplified for 2D CT.

        * candidates = the `top_frac` fraction of Gaussians with the largest mean
          positional-gradient norm (RELATIVE threshold, so it does not depend on
          the loss scale).
        * candidate with max-scale <= split_px -> CLONE (density halved on both copies)
        * candidate with max-scale  > split_px -> SPLIT into 2 children sampled
          from the parent, scale/1.6, density x1.28 (total mass is preserved)
        * afterwards: prune Gaussians with density < rho_prune.
        Net growth per call is at most `top_frac*N`, capped by `max_n`.
        """
        n, dev = self.n, self.xy.device
        avg = grad_accum / denom.clamp(min=1.0)
        s = self.scales()
        budget = int(min(top_frac * n, max(0, max_n - n)))
        cand = torch.zeros(n, dtype=torch.bool, device=dev)
        if budget > 0:
            cand[torch.topk(avg, budget).indices] = True
        big = s.max(dim=1).values > split_px
        clone_m, split_m = cand & ~big, cand & big

        xy, ls = self.xy.data, self.log_scale.data
        rot, lr_ = self.rot.data, self.log_rho.data.clone()
        lr_[clone_m] -= math.log(2.0)          # clone: halve density of both copies

        def offset(m, factor):                 # sample a point from each Gaussian
            z = torch.randn(int(m.sum()), 2, device=dev) * s[m] * factor
            c, sn = torch.cos(rot[m]), torch.sin(rot[m])
            return torch.stack([z[:, 0] * c - z[:, 1] * sn, z[:, 0] * sn + z[:, 1] * c], 1)

        parts_xy = [xy[~split_m], xy[clone_m] + offset(clone_m, 0.5)]
        parts_ls = [ls[~split_m], ls[clone_m]]
        parts_rot = [rot[~split_m], rot[clone_m]]
        parts_lr = [lr_[~split_m], lr_[clone_m]]
        for _ in range(2):                     # two children per split Gaussian
            parts_xy.append(xy[split_m] + offset(split_m, 1.0))
            parts_ls.append(ls[split_m] - math.log(1.6))
            parts_rot.append(rot[split_m])
            parts_lr.append(lr_[split_m] + math.log(1.6 ** 2 / 2.0))

        new_xy, new_ls = torch.cat(parts_xy), torch.cat(parts_ls)
        new_rot, new_lr = torch.cat(parts_rot), torch.cat(parts_lr)
        keep = torch.exp(new_lr) > rho_prune
        if keep.sum() < 100:                   # safety net
            keep[:] = True
        self._set(new_xy[keep], new_ls[keep], new_rot[keep], new_lr[keep])
        return dict(cloned=int(clone_m.sum()), split=int(split_m.sum()),
                    pruned=int((~keep).sum()), n=self.n)

### Initialisation

In [ ]:
def density_reference(img, size=888, pct=99.5):
    """A robust 'tissue density' scale of an image (used to make the TV term and
    the pruning threshold independent of the sinogram scale)."""
    c = size // 2
    yy, xx = np.mgrid[:size, :size] - c
    inside = (xx ** 2 + yy ** 2) <= c ** 2
    v = img[inside & (img > 0)]
    return float(np.percentile(v, pct)) if v.size else 1.0


def init_from_image(img, n, size=888, uniform_frac=0.1, gamma=0.5, seed=0, s_min=0.5, s_max=60.0):
    """Sample Gaussian centres from a (sparse-view FBP) image - R2-Gaussian style.

    img : 2D float array in the units the model will be trained in.
    Points are drawn with probability ~ max(img,0)^gamma inside the head
    (gamma<1 flattens the bone/soft-tissue contrast so soft tissue gets points too).
    `uniform_frac` of the points are scattered uniformly in the circle so that
    streak artefacts in air can still be 'explained' (and pruned if useless).
    """
    rng = np.random.default_rng(seed)
    S, c = size, size // 2
    yy, xx = np.mgrid[:S, :S] - c
    inside = (xx ** 2 + yy ** 2) <= c ** 2
    pos = np.clip(img, 0, None) * inside
    pos = np.where(pos > 0.05 * pos.max(), pos, 0.0)        # drop air / streak noise
    area = max(int((pos > 0).sum()), 1)

    n_f = int(round(n * (1 - uniform_frac)))
    n_u = n - n_f
    w = pos.ravel() ** gamma
    w /= w.sum()
    flat = rng.choice(S * S, size=n_f, replace=True, p=w)
    py, px = np.unravel_index(flat, (S, S))
    x_f = px - c + rng.uniform(-0.5, 0.5, n_f)
    y_f = py - c + rng.uniform(-0.5, 0.5, n_f)
    r = 0.95 * c * np.sqrt(rng.uniform(size=n_u))
    a = TWO_PI * rng.uniform(size=n_u)
    x = np.concatenate([x_f, r * np.cos(a)])
    y = np.concatenate([y_f, r * np.sin(a)])

    ix = np.clip(np.round(x + c).astype(int), 0, S - 1)
    iy = np.clip(np.round(y + c).astype(int), 0, S - 1)
    val = np.clip(img[iy, ix], 0.01 * img.max(), None)      # density floor = 1% of peak
    a2 = area / n                                            # image area per Gaussian
    s0 = float(np.clip(0.4 * np.sqrt(a2), 1.0, 6.0))         # initial isotropic scale
    rho0 = val * a2 / (TWO_PI * s0 ** 2)                     # ~ val when s0 = 0.4*sqrt(a2)

    t = lambda v: torch.tensor(v, dtype=torch.float32)
    return GaussianCT(
        xy=t(np.stack([x, y], 1)),
        log_scale=torch.full((n, 2), math.log(s0)),
        rot=t(rng.uniform(0, math.pi, n)),
        log_rho=t(np.log(rho0)),
        size=size, s_min=s_min, s_max=s_max,
    )


def init_random(n, size=888, seed=0, s0=3.0, rho0=1e-3, s_min=0.5, s_max=60.0):
    """Ablation: uniform random initialisation inside the reconstruction circle.
    Density is calibrated afterwards by train_3dgs.calibrate_density()."""
    rng = np.random.default_rng(seed)
    c = size // 2
    r = 0.95 * c * np.sqrt(rng.uniform(size=n))
    a = TWO_PI * rng.uniform(size=n)
    xy = np.stack([r * np.cos(a), r * np.sin(a)], 1)
    t = lambda v: torch.tensor(v, dtype=torch.float32)
    return GaussianCT(
        xy=t(xy),
        log_scale=torch.full((n, 2), math.log(s0)),
        rot=t(rng.uniform(0, math.pi, n)),
        log_rho=torch.full((n,), math.log(rho0)),
        size=size, s_min=s_min, s_max=s_max,
    )

## 4. Training and main loop

In [11]:
def make_optimizer(model, cfg):
    groups = [
        {"params": [model.xy], "lr": cfg.lr_xy},
        {"params": [model.log_scale], "lr": cfg.lr_scale},
        {"params": [model.rot], "lr": cfg.lr_rot},
        {"params": [model.log_rho], "lr": cfg.lr_rho},
    ]
    for g in groups:
        g["base_lr"] = g["lr"]
    return torch.optim.Adam(groups, eps=1e-15)


@torch.no_grad()
def calibrate_density(model, theta, det, target, cfg):
    """Scale all densities so the initial projections have the same total mass as the
    measured sparse sinogram (cheap fix for the init-density approximation)."""
    pred_sum = 0.0
    for i in range(0, len(theta), 16):
        pred_sum += float(model.project(theta[i:i + 16], det, cfg.gauss_chunk, False).sum())
    alpha = float(target.sum()) / max(pred_sum, 1e-12)
    model.log_rho.data += float(np.log(max(alpha, 1e-6)))
    return alpha


def tv_patch_loss(model, patch, rho_ref, chunk):
    """Total variation of a random patch of the rendered image (differentiable).
    Normalised by rho_ref so the weight does not depend on the sinogram scale."""
    c = model.c
    r = 0.8 * c * np.sqrt(np.random.rand())
    a = 2 * np.pi * np.random.rand()
    cx, cy = int(r * np.cos(a)), int(r * np.sin(a))
    dev = model.xy.device
    ar = torch.arange(patch, device=dev, dtype=torch.float32) - patch // 2
    img = model.render_region(ar + cy, ar + cx, chunk=chunk) / rho_ref
    return (img[1:] - img[:-1]).abs().mean() + (img[:, 1:] - img[:, :-1]).abs().mean()

In [ ]:
def fit_one(img_in, cfg, device, gt=None, preview_path=None):
    """Reconstruct one slice. img_in: sparse-view PNG as float [0,1] (888x888).
    Returns dict with the reconstruction `rec` (float [0,1], same intensity convention as the input PNG)."""
    set_seed(cfg.seed)
    sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
    sync()
    t0 = time.perf_counter()

    idx = view_indices(cfg.n_views_input)              # the 98 angles behind the sparse PNGs
    theta_deg = THETA_FULL_DEG[idx]
    b, img, sino = png_to_sinogram(img_in, theta_deg)  # PNG -> background-free image + pseudo-sinogram
    n_views = sino.shape[1]
    scale = float(sino.max()) or 1.0                   # normalise for optimisation (undone when rendering)
    img_n = img / scale                                # radon(img_n) == sino / scale
    rho_ref = density_reference(img_n, OUTPUT_SIZE)

    if cfg.init == "image":
        model = init_from_image(img_n, cfg.n_init, OUTPUT_SIZE, cfg.init_uniform_frac,
                                cfg.init_gamma, cfg.seed, cfg.s_min, cfg.s_max)
    else:
        model = init_random(cfg.n_init, OUTPUT_SIZE, cfg.seed, s_min=cfg.s_min, s_max=cfg.s_max)
    model = model.to(device)

    det = torch.arange(N_DET, device=device, dtype=torch.float32) - N_DET // 2
    theta = torch.tensor(np.deg2rad(theta_deg) * cfg.angle_sign, dtype=torch.float32, device=device)
    target = torch.tensor((sino / scale).T, dtype=torch.float32, device=device)     # (V, D)
    alpha = calibrate_density(model, theta, det, target, cfg)

    opt = make_optimizer(model, cfg)
    V = min(cfg.views_per_step, n_views)
    d_from, d_until = int(cfg.densify_from * cfg.iters), int(cfg.densify_until * cfg.iters)
    gacc, gden = torch.zeros(model.n, device=device), torch.zeros(model.n, device=device)
    loss_val, t_loop = float("nan"), time.perf_counter()

    for it in range(1, cfg.iters + 1):
        decay = cfg.lr_final_ratio ** (it / cfg.iters)
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * decay

        sel = torch.randperm(n_views, device=device)[:V]
        pred = model.project(theta[sel], det, cfg.gauss_chunk, cfg.use_checkpoint)
        err = pred - target[sel]
        loss = err.abs().mean()
        if cfg.l2_weight > 0:
            loss = loss + cfg.l2_weight * (err ** 2).mean()
        if cfg.tv_weight > 0:
            loss = loss + cfg.tv_weight * tv_patch_loss(model, cfg.tv_patch, rho_ref, cfg.tv_chunk)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        in_window = d_from <= it <= d_until
        if in_window and model.xy.grad is not None:
            gacc += model.xy.grad.norm(dim=-1)
            gden += 1
        opt.step()
        with torch.no_grad():
            model.xy.data.clamp_(-model.c, model.c)

        if in_window and it % cfg.densify_every == 0:
            info = model.densify_and_prune(gacc, gden, cfg.densify_top_frac, cfg.split_px,
                                           cfg.prune_rho_rel * rho_ref, cfg.max_gaussians)
            opt = make_optimizer(model, cfg)   
            gacc, gden = torch.zeros(model.n, device=device), torch.zeros(model.n, device=device)
            print(f"    it {it}: densify {info}")

        if it % cfg.log_every == 0 or it == cfg.iters:
            loss_val = float(loss)
            print(f"    it {it:5d}/{cfg.iters} | loss {loss_val:.5f} | N={model.n} | "
                  f"{(time.perf_counter() - t_loop) / it:.3f} s/it")

    rec_n = model.render_image(chunk=64)
    rec = rec_n.cpu().numpy().astype(np.float32) * scale + b      # back to PNG units, background restored
    rec = np.clip(rec, 0.0, 1.0).astype(np.float32)
    sync()
    runtime = time.perf_counter() - t0       # PNG->sinogram + init + fit + render (evaluation not counted)

    out = dict(rec=rec, runtime=runtime, n_gauss=model.n, loss=loss_val, alpha=alpha, bg=b)

    # view synthesis: projections at angles NOT used for fitting vs. projections of the GT image 
    if cfg.view_synthesis and gt is not None:
        held = np.setdiff1d(np.arange(N_ANGLES // 2), idx)          # unique half-circle angles
        held = held[np.unique(np.round(np.linspace(0, len(held) - 1, cfg.n_heldout)).astype(int))]
        _, _, gt_sino = png_to_sinogram(gt, THETA_FULL_DEG[held])   # (D, n_held)
        th_h = torch.tensor(np.deg2rad(THETA_FULL_DEG[held]) * cfg.angle_sign,
                            dtype=torch.float32, device=device)
        with torch.no_grad():
            cols = [model.project(th_h[i:i + 16], det, cfg.gauss_chunk, False)
                    for i in range(0, len(th_h), 16)]
        pred_sino = torch.cat(cols).cpu().numpy().T * scale
        # the sparse and the full PNG are normalised separately -> match the global gain first
        gain = float((pred_sino * gt_sino).sum() / max(float((pred_sino ** 2).sum()), 1e-12))
        g_n, p_n = metrics.normalise_pair(gt_sino, pred_sino * gain)
        out.update(proj_psnr=metrics.psnr(p_n, g_n), proj_ssim=metrics.ssim(p_n, g_n),
                   proj_rmse=metrics.rmse(p_n, g_n))

    if preview_path is not None and gt is not None:
        save_preview(preview_path, gt, img_in, rec)

    del model, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [13]:
def save_preview(path, gt, fbp, rec):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _, f_n = metrics.normalise_pair(gt, fbp)
    g_n, r_n = metrics.normalise_pair(gt, rec)
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
    for a, im, t in zip(ax[:3], [f_n, r_n, g_n], ["sparse FBP", "3DGS", "ground truth (full FBP)"]):
        a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
    ax[3].imshow(np.abs(r_n - g_n), cmap="magma", vmin=0, vmax=0.3); ax[3].set_title("|3DGS - GT|")
    ax[3].axis("off")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.close(fig)

In [ ]:
def main(cfg):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        print("[warn] no GPU found - this will be extremely slow. Use a GPU runtime.")
    if not cfg.series:
        raise SystemExit("cfg.series is empty.")
    out_dir = Path(cfg.out_dir)
    csv_name = "3dgs_results.csv"
    tag = cfg.tag or ("smoke" if cfg.smoke else "")
    if tag:   # smoke / tuning runs never touch the official outputs
        out_dir, csv_name = out_dir / f"_{tag}", f"3dgs_{tag}.csv"
    out_dir.mkdir(parents=True, exist_ok=True)
    csv = out_dir / csv_name

    done = set()
    if csv.exists():
        d = pd.read_csv(csv)
        done = set(zip(d["series"], d["slice"]))

    for sid in cfg.series:
        items = list_slices_png(cfg, sid)
        if not items:
            print(f"[warn] {sid}: no slice with an input PNG AND a ground-truth PNG - check the folders")
            continue
        if cfg.smoke:
            items = items[:1]
        elif cfg.max_slices:
            items = items[:cfg.max_slices]
        for s, f_in, f_gt in items:
            if (sid, s) in done:
                print(f"{sid} slice {s:02d}: already done, skipped"); continue
            print(f"{sid} slice {s:02d}")
            img_in, gt = load_png(f_in), load_png(f_gt)
            name = f"{sid}_slice{s:02d}_v{cfg.n_views_input:03d}.png"
            prev = out_dir / "previews" / name if cfg.save_preview else None
            r = fit_one(img_in, cfg, device, gt, prev)
            save_png(out_dir / name, r["rec"])
            rec8 = load_png(out_dir / name)             # score what was actually saved (8-bit)
            m, m_in = metrics.evaluate(rec8, gt), metrics.evaluate(img_in, gt)
            row = {"series": sid, "slice": s, "method": "3DGS", "views": cfg.n_views_input,
                   "runtime_s": r["runtime"], **m,
                   "proj_psnr": r.get("proj_psnr"), "proj_ssim": r.get("proj_ssim"),
                   "proj_rmse": r.get("proj_rmse"), "n_gaussians": r["n_gauss"],
                   "final_loss": r["loss"], "iters": cfg.iters, "seed": cfg.seed, "bg_level": r["bg"],
                   "input_psnr": m_in["psnr"], "input_ssim": m_in["ssim"]}
            pd.DataFrame([row]).to_csv(csv, mode="a", header=not csv.exists(), index=False)
            print(f"  -> PSNR {m['psnr']:.2f} (sparse input {m_in['psnr']:.2f}) | SSIM {m['ssim']:.3f} "
                  f"(input {m_in['ssim']:.3f}) | head-PSNR {m['psnr_head']:.2f} | "
                  f"held-out proj PSNR {r.get('proj_psnr', float('nan')):.2f} | {r['runtime']:.0f} s")

    if csv.exists():
        res = pd.read_csv(csv)
        cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head",
                "proj_psnr", "proj_ssim", "runtime_s"]
        summary = res.groupby("views")[cols].agg(["mean", "std"]).round(3)
        summary.to_csv(out_dir / csv_name.replace(".csv", "_summary.csv"))
        print(summary)

## 5. Geometry check 

In [ ]:
def sanity_check(cfg, device):
    S, c = N_DET, N_DET // 2
    t = lambda v: torch.tensor(v, dtype=torch.float32)
    model = GaussianCT(t([[0, 0], [60, -40], [-100, 80]]),
                       t([[10, 5], [4, 12], [15, 15]]).log(), t([0.3, 1.0, 0.0]),
                       t([1.0, 2.0, 0.5]).log(), size=S).to(device)
    img = model.render_image(chunk=3).cpu().numpy()
    theta_deg = np.array([0, 17, 45, 90, 133, 171], dtype=float)
    ref = radon(img, theta=theta_deg, circle=True)             # skimage forward projector
    det = torch.arange(S, dtype=torch.float32, device=device) - c
    print("[1] Gaussian projector vs skimage.radon of the rendered image")
    errs = {}
    for sign in (1.0, -1.0):
        th = torch.tensor(np.deg2rad(theta_deg) * sign, dtype=torch.float32, device=device)
        p = model.project(th, det, chunk=None).detach().cpu().numpy().T
        errs[sign] = np.linalg.norm(p - ref) / np.linalg.norm(ref)
        print(f"    angle_sign {sign:+.0f}: relative error {errs[sign]:.3%}")
    best = min(errs, key=errs.get)
    print(f"    -> use angle_sign = {best:+.0f}" + ("" if errs[best] < 0.05 else
          "   [WARNING: error still > 5% - the projector/renderer geometry is off, do not run!]"))

    print("[2] PNG input: background removal + pseudo-sinogram (needs data paths)")
    try:
        sid = cfg.series[0]
        s, f_in, f_gt = list_slices_png(cfg, sid)[0]
        img_in = load_png(f_in)
        idx = view_indices(cfg.n_views_input)
        b, img, sino = png_to_sinogram(img_in, THETA_FULL_DEG[idx])
        sat = float((img_in >= 0.999).mean())
        print(f"    {sid} slice {s}: background level b = {b:.3f} ({b * 255:.0f}/255) | "
              f"saturated pixels {sat:.3%} | pseudo-sinogram {sino.shape}, max {sino.max():.2f}")
        print("    expectation: b about 0.06, saturated pixels ~0 %, sinogram (888, 98).")
        if sat > 0.01:
            print("    [WARNING] >1 % of the pixels are clipped at 255 - bone intensity is lost in the PNG")
    except Exception as e:  # noqa: BLE001
        print(f"    skipped ({type(e).__name__}: {e}) - check the data paths in the config and that the sparse PNGs exist")

In [16]:
sanity_check(replace(cfg, series=VAL_SERIES[:1]), device)

[1] Gaussian projector vs skimage.radon of the rendered image
    angle_sign +1: relative error 0.077%
    angle_sign -1: relative error 108.509%
    -> use angle_sign = +1
[2] PNG input: background removal + pseudo-sinogram (needs data paths)
    series_1050 slice 0: background level b = 0.196 (50/255) | saturated pixels 0.000% | pseudo-sinogram (888, 98), max 113.52
    expectation: b about 0.06, saturated pixels ~0 %, sinogram (888, 98).


## 6. Smoke test (1 slice of a validation patient, 300 iterations)

In [17]:
smoke_cfg = replace(cfg, series=VAL_SERIES[:1], smoke=True, iters=300, n_init=5000,
                    log_every=50, densify_every=50)
main(smoke_cfg)

series_1050 slice 00
    it 50: densify {'cloned': 250, 'split': 0, 'pruned': 0, 'n': 5250}
    it    50/300 | loss 0.00949 | N=5250 | 0.032 s/it


/tmp/ipykernel_31863/3972830502.py:67: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  loss_val = float(loss)


    it 100: densify {'cloned': 262, 'split': 0, 'pruned': 0, 'n': 5512}
    it   100/300 | loss 0.00449 | N=5512 | 0.031 s/it
    it 150: densify {'cloned': 275, 'split': 0, 'pruned': 0, 'n': 5787}
    it   150/300 | loss 0.00389 | N=5787 | 0.031 s/it
    it   200/300 | loss 0.00320 | N=5787 | 0.032 s/it
    it   250/300 | loss 0.00317 | N=5787 | 0.032 s/it
    it   300/300 | loss 0.00320 | N=5787 | 0.032 s/it
  -> PSNR 34.49 (sparse input 45.82) | SSIM 0.931 (input 0.976) | head-PSNR 25.31 | held-out proj PSNR 38.97 | 16 s
         psnr       ssim       rmse     psnr_head     ssim_head     rmse_head  \
         mean std   mean std   mean std      mean std      mean std      mean   
views                                                                           
98     34.487 NaN  0.931 NaN  0.019 NaN    25.313 NaN     0.445 NaN     0.054   

          proj_psnr     proj_ssim     runtime_s      
      std      mean std      mean std      mean std  
views                                

## 7. Tuning on the validation patients

In [ ]:
tune_cfg = replace(cfg, series=VAL_SERIES[:1], max_slices=1, iters=400, n_init=5000, max_gaussians=10000, tv_weight=0.0, view_synthesis=False, save_preview=False)   # new tag for every variant
main(tune_cfg)

series_1050 slice 00
    it 100: densify {'cloned': 250, 'split': 0, 'pruned': 0, 'n': 5250}
    it 200: densify {'cloned': 262, 'split': 0, 'pruned': 0, 'n': 5512}
    it   250/400 | loss 0.00264 | N=5512 | 0.026 s/it
    it   400/400 | loss 0.00254 | N=5512 | 0.027 s/it
  -> PSNR 34.77 (sparse input 45.82) | SSIM 0.934 (input 0.976) | head-PSNR 25.59 | held-out proj PSNR nan | 17 s
         psnr       ssim       rmse     psnr_head     ssim_head     rmse_head  \
         mean std   mean std   mean std      mean std      mean std      mean   
views                                                                           
98     34.774 NaN  0.934 NaN  0.018 NaN    25.591 NaN     0.464 NaN     0.053   

          proj_psnr     proj_ssim     runtime_s      
      std      mean std      mean std      mean std  
views                                                
98    NaN       NaN NaN       NaN NaN    17.155 NaN  


## 8. Full run (official result)

In [ ]:
assert cfg.series, "set TEST_SERIES to a list of series IDs to run the full evaluation"
main(cfg)

series_1030 slice 00
    it 100: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it 200: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it   250/500 | loss 0.00258 | N=15000 | 0.084 s/it
    it 300: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it   500/500 | loss 0.00194 | N=15000 | 0.084 s/it
  -> PSNR 33.78 (sparse input 47.95) | SSIM 0.937 (input 0.983) | head-PSNR 22.74 | held-out proj PSNR 41.21 | 51 s
series_1030 slice 04
    it 100: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it 200: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it   250/500 | loss 0.00190 | N=15000 | 0.084 s/it
    it 300: densify {'cloned': 0, 'split': 0, 'pruned': 0, 'n': 15000}
    it   500/500 | loss 0.00124 | N=15000 | 0.084 s/it
  -> PSNR 32.63 (sparse input 48.08) | SSIM 0.935 (input 0.984) | head-PSNR 21.84 | held-out proj PSNR 43.21 | 51 s
series_1030 slice 08
    it 100: densify {'cloned': 0, 'split': 0, 'pruned': 0, 

## 9. Results

In [20]:
CSV = Path(cfg.out_dir) / "3dgs_results.csv"
res = pd.read_csv(CSV)
cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head", "proj_psnr", "proj_ssim", "runtime_s"]
summary = res.groupby("views")[cols].agg(["mean", "std"]).round(3)
summary.to_csv(Path(cfg.out_dir) / "3dgs_results_summary.csv")
print(f"{res.series.nunique()} series, {len(res)} slices, median {res.n_gaussians.median():.0f} Gaussians")
summary

10 series, 93 slices, median 15000 Gaussians


psnr         ssim          rmse        psnr_head        ssim_head  \
         mean    std  mean    std   mean    std      mean    std      mean   
views                                                                        
98     33.042  4.244  0.93  0.025  0.024  0.008    22.375  3.692     0.358   

             rmse_head        proj_psnr        proj_ssim        runtime_s  \
         std      mean    std      mean    std      mean    std      mean   
views                                                                       
98     0.173     0.081  0.024    40.813  2.512     0.852  0.084    51.069   

              
         std  
views         
98     3.556

In [21]:
# 3DGS vs. the sparse input PNG itself (= what we start from); above the dashed line = 3DGS improved the slice
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (c3, ci, lab) in zip(ax, [("psnr", "input_psnr", "PSNR [dB]"), ("ssim", "input_ssim", "SSIM")]):
    a.scatter(res[ci], res[c3], s=22)
    lo, hi = min(res[ci].min(), res[c3].min()), max(res[ci].max(), res[c3].max())
    a.plot([lo, hi], [lo, hi], "k--", lw=1)
    a.set_xlabel(f"sparse input (98 views) {lab}"); a.set_ylabel(f"3DGS {lab}"); a.grid(alpha=.3)
    a.set_title("each dot = one slice")
plt.tight_layout(); plt.show()
print("mean gain over the sparse input: PSNR %+.2f dB, SSIM %+.3f" % ((res.psnr - res.input_psnr).mean(),
                                                                    (res.ssim - res.input_ssim).mean()))

mean gain over the sparse input: PSNR -15.09 dB, SSIM -0.054


## 10. One slice (figure for the report)

In [22]:
row0 = res.iloc[0]
sid, s = row0.series, int(row0.slice)
f_in, f_gt = {a: (b_, c_) for a, b_, c_ in list_slices_png(cfg, sid)}[s]
inp, gt = load_png(f_in), load_png(f_gt)
rec = load_png(Path(cfg.out_dir) / f"{sid}_slice{s:02d}_v{cfg.n_views_input:03d}.png")

fig, ax = plt.subplots(1, 4, figsize=(15, 3.9))
for a, im, t in zip(ax[:3], [gt, inp, rec], ["ground truth (full view)", f"sparse input ({cfg.n_views_input} views)", "3DGS"]):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t)
ax[3].imshow(np.abs(rec - gt), cmap="magma", vmin=0, vmax=0.3); ax[3].set_title("|3DGS - ground truth|")
for a in ax: a.axis("off")
plt.suptitle(f"{sid}, slice {s}"); plt.tight_layout(); plt.show()